# NB05 — Advanced Fine-Tuning (5-class, robustness-regularized)

Single-task 5-class fine-tuning of BanglishBERT / MuRIL / XLM-R with the framework in plans.md §5:
**FGM adversarial training only · (R-Drop · EMA weights · multi-sample dropout · class-balanced focal->are turned off)·
balanced sampler**, script-aware (BanglaBERT ← Bangla script only). Uniform LR (no decay).

Trains on the 70% split, early-stops on the 10% val, **reports on the pure 20% test**. Saves per-model
val+test logits for the ensemble (NB06). Every technique is a config toggle → feeds the NB08 ablation.


In [42]:
import os
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

In [43]:
# ── Install (uncomment if env differs) ──
#!pip install torch==2.4.1 transformers==4.44.2 scikit-learn==1.5.1 pandas==2.2.2 numpy==1.26.4 sentencepiece==0.2.0 --quiet
print("deps assumed present")

deps assumed present


In [44]:
import os, json, time, math, random, warnings
from collections import defaultdict
import numpy as np, pandas as pd
import torch, torch.nn as nn, torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
from transformers import AutoModel, AutoTokenizer, get_linear_schedule_with_warmup
from sklearn.metrics import (f1_score, accuracy_score, matthews_corrcoef, roc_auc_score)
warnings.filterwarnings("ignore")
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("CUDA:", torch.cuda.is_available(),
      "| GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU")

CUDA: True | GPU: NVIDIA L40S


In [45]:
# ═══════════════════════════════════════════════════════════════════
# CONFIG — advanced 5-class. Every technique is a toggle (ablation-ready).
# ═══════════════════════════════════════════════════════════════════
DEBUG         = False        # smoke-test first; set False for the real run
DEBUG_SAMPLES = 400
DEBUG_EPOCHS  = 2

CONFIG = {
    "models": {
    "banglishbert": "csebuetnlp/banglishbert",
    "banglabert":   "csebuetnlp/banglabert",
    "muril":        "google/muril-base-cased",
    "xlmr":         "xlm-roberta-base",
    },
    "max_length": 128, "batch_size": 32, "eval_batch_size": 128, "grad_accum_steps": 1,
    "num_workers": 0, "epochs": 8, "encoder_lr": 2e-5, "head_lr": 8e-5,
    "weight_decay": 0.01, "warmup_ratio": 0.10, "max_grad_norm": 1.0,
    "label_smoothing": 0.03, "focal_gamma": 2.0, "class_weight_beta": 0.999,
    "dropout": 0.25, "head_hidden_dim": 384, "n_msd": 1, "patience": 3,
    "use_fp16": torch.cuda.is_available(),
    # ── advanced toggles ──
    "use_focal": False, "use_cw": False,
    "use_balanced_sampler": False, "sampler_alpha": 0.5,
    "per_class_loss_multiplier": {},
    "use_fgm":   True, "fgm_epsilon": 1.0,
    "use_rdrop": False, "rdrop_alpha": 0.5,
    "use_ema":   False, "ema_decay": 0.999,
    "banglabert_script_only": True,
    "banglabert_script_key": "banglabert",
}
USE_LRDECAY = False   # constraint 7

LABEL_COL = "label5"
TASK = "label5"
TEXT_COL = "text_clean"
SPLIT_DIR = "../data/splits"
OUTPUT_DIR = "../outputs/models_main"
RUN_MODELS = ["banglishbert", "banglabert", "muril", "xlmr"]
RUN_SEEDS  = [42, 123, 456]
FORCE_RETRAIN = False
os.makedirs(OUTPUT_DIR, exist_ok=True)
if DEBUG:
    CONFIG["epochs"] = DEBUG_EPOCHS; RUN_SEEDS = [42]
    print("⚠ DEBUG mode")
print(f"toggles: sampler={CONFIG['use_balanced_sampler']} fgm={CONFIG['use_fgm']} "
      f"rdrop={CONFIG['use_rdrop']} ema={CONFIG['use_ema']} script_aware={CONFIG['banglabert_script_only']}")
print(f"runs: {len(RUN_MODELS)*len(RUN_SEEDS)} | LRdecay={USE_LRDECAY}")

toggles: sampler=False fgm=True rdrop=False ema=False script_aware=True
runs: 12 | LRdecay=False


In [46]:
# ── Load splits + build label encoder ───────────────────────────────────────
train_full = pd.read_csv(f"{SPLIT_DIR}/random_train.csv")
val_df     = pd.read_csv(f"{SPLIT_DIR}/random_val.csv")
test_df    = pd.read_csv(f"{SPLIT_DIR}/random_test.csv")
classes = sorted(train_full[LABEL_COL].unique())
label_enc = {c: i for i, c in enumerate(classes)}
NUM_CLASSES = len(classes)
assert NUM_CLASSES == 5, f"expected 5 classes, got {NUM_CLASSES}"
print(f"classes ({NUM_CLASSES}): {label_enc}")
print(f"train={len(train_full):,} val={len(val_df):,} test={len(test_df):,} (test = pure 20%)")
if DEBUG:
    train_full = train_full.groupby(LABEL_COL, group_keys=False).apply(
        lambda g: g.sample(min(len(g), DEBUG_SAMPLES//NUM_CLASSES), random_state=42))
    val_df  = val_df.sample(min(len(val_df), 300), random_state=42)
    test_df = test_df.sample(min(len(test_df), 300), random_state=42)
with open(f"{OUTPUT_DIR}/label_encoder.json", "w") as f:
    json.dump(label_enc, f, indent=2)

classes (5): {'abusive': 0, 'none': 1, 'religious': 2, 'sexual': 3, 'threat': 4}
train=66,026 val=9,432 test=18,865 (test = pure 20%)


In [47]:
# ── Dataset / Collator ──────────────────────────────────────────────────────
class DS(Dataset):
    def __init__(self, df, tok, maxlen, enc):
        self.texts  = df.reset_index(drop=True)[TEXT_COL].fillna("").astype(str).tolist()
        self.labels = [int(enc.get(v, -1)) for v in df[LABEL_COL]]
        self.tok, self.maxlen = tok, maxlen
    def __len__(self): return len(self.texts)
    def __getitem__(self, i):
        e = self.tok(self.texts[i], max_length=self.maxlen, truncation=True, padding=False)
        item = {k: torch.tensor(v, dtype=torch.long) for k, v in e.items()}
        item["label"] = torch.tensor(self.labels[i], dtype=torch.long)
        return item

class Collator:
    def __init__(self, tok): self.tok = tok
    def __call__(self, fs):
        labels = torch.stack([f["label"] for f in fs])
        feats = [{k: v for k, v in f.items() if k != "label"} for f in fs]
        b = self.tok.pad(feats, padding=True, return_tensors="pt"); b["label"] = labels
        return b
print("dataset ok")

dataset ok


In [48]:
# ── Loss, class weights, sampler ────────────────────────────────────────────
class FocalLoss(nn.Module):
    def __init__(self, gamma=2.0, weight=None, label_smoothing=0.0):
        super().__init__(); self.gamma, self.weight, self.ls = gamma, weight, label_smoothing
    def forward(self, logits, t):
        ce = F.cross_entropy(logits, t, weight=self.weight, reduction="none", label_smoothing=self.ls)
        return (((1 - torch.exp(-ce)) ** self.gamma) * ce).mean()

def build_class_weights(series, enc, beta=0.999, max_w=10.0, per_class_mult=None):
    mapped = series.map(enc).dropna().astype(int); n = len(enc)
    counts = mapped.value_counts().sort_index(); w = np.ones(n, dtype=np.float32)
    for i in range(n):
        c = int(counts.get(i, 0))
        if c > 0: w[i] = 1.0 / max((1.0 - beta**c) / max(1.0 - beta, 1e-12), 1e-9)
    w = np.clip(w, w.min(), w.min() * max_w)
    if per_class_mult:
        for cname, m in per_class_mult.items():
            if cname in enc: w[enc[cname]] *= float(m)
    w = w / w.mean()
    return torch.tensor(w, dtype=torch.float32, device=device)

def make_balanced_sampler(df, enc, alpha=0.5):
    y = df[LABEL_COL].map(enc).fillna(-1).astype(int).values
    counts = np.bincount(y[y >= 0], minlength=len(enc)).astype(float); counts[counts == 0] = 1.0
    cw = (1.0 / counts) ** float(alpha)
    sw = np.where(y >= 0, cw[np.clip(y, 0, None)], 0.0)
    return WeightedRandomSampler(torch.as_tensor(sw, dtype=torch.double), len(sw), replacement=True)
print("loss/sampler ok")

loss/sampler ok


In [49]:
# ── Model: encoder + multi-sample-dropout head ──────────────────────────────
class MSDHead(nn.Module):
    def __init__(self, hidden, n_cls, dropout=0.25, inner=384, n_msd=4):
        super().__init__(); inner = min(inner, hidden)
        self.pre = nn.Sequential(nn.Linear(hidden, inner), nn.GELU(), nn.LayerNorm(inner))
        self.drops = nn.ModuleList([nn.Dropout(dropout) for _ in range(max(1, n_msd))])
        self.out = nn.Linear(inner, n_cls)
    def forward(self, x):
        h = self.pre(x)
        if self.training and len(self.drops) > 1:
            return torch.stack([self.out(d(h)) for d in self.drops], 0).mean(0)
        return self.out(self.drops[0](h))

class Encoder(nn.Module):
    def __init__(self, name, n_cls, dropout=0.25, inner=384, n_msd=4):
        super().__init__(); self.encoder = AutoModel.from_pretrained(name)
        h = self.encoder.config.hidden_size
        self._tti = self.encoder.config.model_type.lower() not in ("roberta", "xlm-roberta")
        self.head = MSDHead(h, n_cls, dropout, inner, n_msd)
    def _pool(self, out, mask):
        hs = out.last_hidden_state; cls = hs[:, 0]
        mean = (hs * mask.unsqueeze(-1).float()).sum(1) / mask.sum(1, keepdim=True).float().clamp(1)
        return 0.5 * cls + 0.5 * mean
    def forward(self, ids, mask, tti=None):
        kw = {"input_ids": ids, "attention_mask": mask}
        if self._tti and tti is not None: kw["token_type_ids"] = tti
        return self.head(self._pool(self.encoder(**kw), mask))

def uniform_params(model, enc_lr, head_lr, wd):
    nd = ["bias", "LayerNorm.weight", "LayerNorm.bias"]; g = []
    for grp, lr in [(model.encoder, enc_lr), (model.head, head_lr)]:
        dec, ndec = [], []
        for n, p in grp.named_parameters():
            if p.requires_grad: (ndec if any(x in n for x in nd) else dec).append(p)
        g += [{"params": dec, "lr": lr, "weight_decay": wd},
              {"params": ndec, "lr": lr, "weight_decay": 0.0}]
    return g
print("model ok")

model ok


In [50]:
# ── FGM adversarial + EMA + R-Drop KL ───────────────────────────────────────
class FGM:
    """Perturb token embeddings along the (normalized) gradient direction."""
    def __init__(self, model, epsilon=1.0, emb_key="word_embeddings"):
        self.model, self.eps, self.key, self.backup = model, epsilon, emb_key, {}
    def attack(self):
        for name, p in self.model.named_parameters():
            if p.requires_grad and self.key in name and p.grad is not None:
                self.backup[name] = p.data.clone()
                norm = torch.norm(p.grad)
                if norm != 0 and not torch.isnan(norm):
                    p.data.add_(self.eps * p.grad / norm)
    def restore(self):
        for name, p in self.model.named_parameters():
            if name in self.backup: p.data = self.backup[name]
        self.backup = {}

class EMA:
    def __init__(self, model, decay=0.999):
        self.decay = decay
        self.shadow = {n: p.detach().clone() for n, p in model.named_parameters() if p.requires_grad}
        self.backup = {}
    def update(self, model):
        for n, p in model.named_parameters():
            if p.requires_grad:
                self.shadow[n].mul_(self.decay).add_(p.detach(), alpha=1 - self.decay)
    def apply_shadow(self, model):
        self.backup = {n: p.detach().clone() for n, p in model.named_parameters() if p.requires_grad}
        for n, p in model.named_parameters():
            if p.requires_grad: p.data.copy_(self.shadow[n])
    def restore(self, model):
        for n, p in model.named_parameters():
            if p.requires_grad and n in self.backup: p.data.copy_(self.backup[n])
        self.backup = {}

def rdrop_kl(lp, lq):
    p_log, q_log = F.log_softmax(lp, -1), F.log_softmax(lq, -1)
    p, q = p_log.exp(), q_log.exp()
    return 0.5 * ((p * (p_log - q_log)).sum(-1) + (q * (q_log - p_log)).sum(-1)).mean()
print("fgm/ema/rdrop ok")

fgm/ema/rdrop ok


In [51]:
# ── Metrics + logits ────────────────────────────────────────────────────────
@torch.no_grad()
def evaluate(model, loader):
    model.eval(); P, Y, PR = [], [], []
    for b in loader:
        b = {k: v.to(device) for k, v in b.items()}
        lg = model(b["input_ids"], b["attention_mask"], b.get("token_type_ids"))
        pr = F.softmax(lg, -1).cpu().numpy(); y = b["label"].cpu().numpy(); m = y >= 0
        P.extend(pr[m].argmax(-1)); Y.extend(y[m]); PR.extend(pr[m])
    y, p, pr = np.array(Y), np.array(P), np.array(PR)
    rec = {"macro_f1": round(float(f1_score(y, p, average="macro", zero_division=0)), 4),
           "weighted_f1": round(float(f1_score(y, p, average="weighted", zero_division=0)), 4),
           "accuracy": round(float(accuracy_score(y, p)), 4),
           "mcc": round(float(matthews_corrcoef(y, p)), 4)}
    try:
        rec["auroc"] = round(float(roc_auc_score(y, pr, multi_class="ovr", average="macro",
                                                 labels=list(range(pr.shape[1])))), 4)
    except Exception:
        rec["auroc"] = float("nan")
    return rec

@torch.no_grad()
def get_logits(model, loader):
    model.eval(); out = []
    for b in loader:
        b = {k: v.to(device) for k, v in b.items()}
        out.append(model(b["input_ids"], b["attention_mask"], b.get("token_type_ids")).cpu())
    return torch.cat(out)
print("metrics ok")

metrics ok


In [52]:
# ── Train one (model, seed) ─────────────────────────────────────────────────
def set_seed(s):
    random.seed(s)
    np.random.seed(s)
    torch.manual_seed(s)
    torch.cuda.manual_seed_all(s)


def train_one(model_key, model_name, seed):
    set_seed(seed)

    sd = f"{OUTPUT_DIR}/{model_key}_seed{seed}"
    os.makedirs(sd, exist_ok=True)

    cfg = CONFIG

    # ── Script-aware data scope ─────────────────────────────────────────────
    # BanglaBERT is a Bangla-script specialist:
    #   train = Bangla only
    #   val   = Bangla only
    #   test  = Bangla only
    #
    # For ensemble compatibility:
    #   saved val_logits/test_logits are still full-size [N, C]
    #   Romanized rows receive neutral zero logits
    #   NB06 will also assign BanglaBERT zero row-weight on Romanized rows.
    is_script_model = (
        model_key == cfg.get("banglabert_script_key", "banglabert")
        and cfg.get("banglabert_script_only", False)
        and "script" in train_full.columns
        and "script" in val_df.columns
        and "script" in test_df.columns
    )

    if is_script_model:
        train_df = (
            train_full[train_full["script"].astype(str).str.lower().eq("bangla")]
            .reset_index(drop=True)
        )

        val_mask = val_df["script"].astype(str).str.lower().eq("bangla").values
        test_mask = test_df["script"].astype(str).str.lower().eq("bangla").values

        val_eval_idx = np.where(val_mask)[0]
        test_eval_idx = np.where(test_mask)[0]

        val_eval_df = val_df.iloc[val_eval_idx].reset_index(drop=True)
        test_eval_df = test_df.iloc[test_eval_idx].reset_index(drop=True)

        if len(train_df) == 0:
            raise ValueError(f"{model_key}: no Bangla-script rows found in train_full.")
        if len(val_eval_df) == 0:
            raise ValueError(f"{model_key}: no Bangla-script rows found in val_df.")
        if len(test_eval_df) == 0:
            raise ValueError(f"{model_key}: no Bangla-script rows found in test_df.")

        print(
            f"  script-aware: {model_key} uses Bangla-only scope | "
            f"train={len(train_df):,}, "
            f"val={len(val_eval_df):,}/{len(val_df):,}, "
            f"test={len(test_eval_df):,}/{len(test_df):,}"
        )

    else:
        train_df = train_full
        val_eval_df = val_df
        test_eval_df = test_df
        val_eval_idx = None
        test_eval_idx = None

    # Optional per-model batch-size override.
    bs = cfg.get("batch_size_override", {}).get(model_key, cfg["batch_size"])
    if bs != cfg["batch_size"]:
        cfg = {
            **cfg,
            "batch_size": bs,
            "grad_accum_steps": cfg["grad_accum_steps"] * max(1, cfg["batch_size"] // bs),
        }

    tok = AutoTokenizer.from_pretrained(model_name)
    coll = Collator(tok)

    lkw = dict(
        collate_fn=coll,
        num_workers=cfg["num_workers"],
        pin_memory=torch.cuda.is_available(),
    )

    train_ds = DS(train_df, tok, cfg["max_length"], label_enc)

    if cfg["use_balanced_sampler"]:
        train_loader = DataLoader(
            train_ds,
            batch_size=cfg["batch_size"],
            sampler=make_balanced_sampler(train_df, label_enc, cfg["sampler_alpha"]),
            shuffle=False,
            drop_last=True,
            **lkw,
        )
    else:
        train_loader = DataLoader(
            train_ds,
            batch_size=cfg["batch_size"],
            shuffle=True,
            drop_last=True,
            **lkw,
        )

    val_loader = DataLoader(
        DS(val_eval_df, tok, cfg["max_length"], label_enc),
        batch_size=cfg["eval_batch_size"],
        shuffle=False,
        **lkw,
    )

    test_loader = DataLoader(
        DS(test_eval_df, tok, cfg["max_length"], label_enc),
        batch_size=cfg["eval_batch_size"],
        shuffle=False,
        **lkw,
    )

    def full_size_logits_from_subset(model, subset_loader, full_n, subset_idx):
        """
        Normal models:
            return logits for all rows.

        Script-aware BanglaBERT:
            compute logits only on Bangla rows, then place them into a full
            [N, C] tensor. Romanized rows receive neutral zero logits.
        """
        raw = get_logits(model, subset_loader)

        if subset_idx is None:
            return raw

        full = torch.zeros((full_n, NUM_CLASSES), dtype=raw.dtype)
        full[torch.as_tensor(subset_idx, dtype=torch.long)] = raw
        return full

    model = Encoder(
        model_name,
        NUM_CLASSES,
        cfg["dropout"],
        cfg["head_hidden_dim"],
        cfg["n_msd"],
    ).to(device)

    cw = build_class_weights(
        train_df[LABEL_COL],
        label_enc,
        beta=cfg["class_weight_beta"],
        per_class_mult=cfg.get("per_class_loss_multiplier", {}),
    ) if cfg.get("use_cw", True) else None

    if cfg.get("use_focal", True):
        focal = FocalLoss(cfg["focal_gamma"], cw, cfg["label_smoothing"])
    else:
        focal = lambda lg, t: F.cross_entropy(lg, t, weight=cw, label_smoothing=cfg["label_smoothing"])

    opt = torch.optim.AdamW(
        uniform_params(
            model,
            cfg["encoder_lr"],
            cfg["head_lr"],
            cfg["weight_decay"],
        )
    )

    nsteps = math.ceil(len(train_loader) / cfg["grad_accum_steps"]) * cfg["epochs"]

    sch = get_linear_schedule_with_warmup(
        opt,
        int(nsteps * cfg["warmup_ratio"]),
        nsteps,
    )

    scaler = (
        torch.amp.GradScaler("cuda")
        if (cfg["use_fp16"] and device.type == "cuda")
        else None
    )

    fgm = FGM(model, cfg["fgm_epsilon"]) if cfg["use_fgm"] else None
    ema = EMA(model, cfg["ema_decay"]) if cfg["use_ema"] else None

    best, noimp = -1.0, 0

    for ep in range(cfg["epochs"]):
        model.train()
        opt.zero_grad(set_to_none=True)

        run, t0 = 0.0, time.time()

        for step, b in enumerate(train_loader, 1):
            b = {k: v.to(device) for k, v in b.items()}
            y = b["label"]

            with torch.autocast(device_type=device.type, enabled=scaler is not None):
                lg1 = model(
                    b["input_ids"],
                    b["attention_mask"],
                    b.get("token_type_ids"),
                )

                if cfg["use_rdrop"]:
                    lg2 = model(
                        b["input_ids"],
                        b["attention_mask"],
                        b.get("token_type_ids"),
                    )

                    loss = (
                        0.5 * (focal(lg1, y) + focal(lg2, y))
                        + cfg["rdrop_alpha"] * rdrop_kl(lg1, lg2)
                    )
                else:
                    loss = focal(lg1, y)

                loss = loss / cfg["grad_accum_steps"]

            if scaler:
                scaler.scale(loss).backward()
            else:
                loss.backward()

            if fgm is not None:
                fgm.attack()

                with torch.autocast(device_type=device.type, enabled=scaler is not None):
                    lg_adv = model(
                        b["input_ids"],
                        b["attention_mask"],
                        b.get("token_type_ids"),
                    )
                    loss_adv = focal(lg_adv, y) / cfg["grad_accum_steps"]

                if scaler:
                    scaler.scale(loss_adv).backward()
                else:
                    loss_adv.backward()

                fgm.restore()

            if step % cfg["grad_accum_steps"] == 0 or step == len(train_loader):
                if scaler:
                    scaler.unscale_(opt)

                torch.nn.utils.clip_grad_norm_(
                    model.parameters(),
                    cfg["max_grad_norm"],
                )

                if scaler:
                    scaler.step(opt)
                    scaler.update()
                else:
                    opt.step()

                sch.step()
                opt.zero_grad(set_to_none=True)

                if ema is not None:
                    ema.update(model)

            run += loss.item() * cfg["grad_accum_steps"]

        # Validate raw, then EMA (after warmup); keep whichever scores higher on val.
        vmac_raw = evaluate(model, val_loader)["macro_f1"]
        vmac, use_ema = vmac_raw, False
        if ema is not None and ep >= 2:
            ema.apply_shadow(model)
            vmac_ema = evaluate(model, val_loader)["macro_f1"]
            if vmac_ema >= vmac_raw:
                vmac, use_ema = vmac_ema, True
            ema.restore(model)
        flag = ""
        if vmac > best:
            best, noimp = vmac, 0
            if use_ema: ema.apply_shadow(model)
            torch.save(model.state_dict(), f"{sd}/best_model.pt")
            if use_ema: ema.restore(model)
            flag = " ✅BEST"
        else:
            noimp += 1

        print(
            f"  Ep{ep + 1:2}/{cfg['epochs']} "
            f"loss={run / max(len(train_loader), 1):.4f} "
            f"val_macroF1={vmac:.4f} "
            f"{(time.time() - t0) / 60:.1f}min{flag}"
        )

        if noimp >= cfg["patience"]:
            print(f"  early stop ep{ep + 1}")
            break

    model.load_state_dict(
        torch.load(
            f"{sd}/best_model.pt",
            map_location=device,
            weights_only=True,
        )
    )

    # For BanglaBERT, this is Bangla-script-only test performance.
    # For all other models, this is full-test performance.
    test_metrics = evaluate(model, test_loader)

    # Save full-size logits for NB06.
    # For BanglaBERT, Romanized rows are neutral zero logits.
    val_logits_full = full_size_logits_from_subset(
        model,
        val_loader,
        full_n=len(val_df),
        subset_idx=val_eval_idx,
    )

    test_logits_full = full_size_logits_from_subset(
        model,
        test_loader,
        full_n=len(test_df),
        subset_idx=test_eval_idx,
    )

    torch.save(val_logits_full, f"{sd}/val_logits.pt")
    torch.save(test_logits_full, f"{sd}/test_logits.pt")

    res = {
        "model": model_key,
        "seed": seed,
        "eval_scope": "bangla_only" if is_script_model else "full",
        "best_val_macro_f1": round(best, 4),
        "test_metrics": test_metrics,
        "n_train_rows": int(len(train_df)),
        "n_val_eval_rows": int(len(val_eval_df)),
        "n_test_eval_rows": int(len(test_eval_df)),
        "n_val_total_rows": int(len(val_df)),
        "n_test_total_rows": int(len(test_df)),
    }

    json.dump(res, open(f"{sd}/results.json", "w"), indent=2)

    scope = "Bangla-only 20% TEST" if is_script_model else "20% TEST"

    print(
        f"  → [{scope}] "
        f"macroF1={test_metrics['macro_f1']:.4f} "
        f"wF1={test_metrics['weighted_f1']:.4f} "
        f"acc={test_metrics['accuracy']:.4f} "
        f"mcc={test_metrics['mcc']:.4f}"
    )

    return res


print("trainer ok")

trainer ok


In [53]:
# Salvage muril_seed42 from its epoch-4 checkpoint → mark it complete
sd = f"{OUTPUT_DIR}/muril_seed42"
if os.path.exists(f"{sd}/best_model.pt") and not os.path.exists(f"{sd}/results.json"):
    name = CONFIG["models"]["muril"]; tok = AutoTokenizer.from_pretrained(name); coll = Collator(tok)
    lkw = dict(collate_fn=coll, num_workers=0)
    vl  = DataLoader(DS(val_df,  tok, CONFIG["max_length"], label_enc), batch_size=CONFIG["eval_batch_size"], shuffle=False, **lkw)
    tel = DataLoader(DS(test_df, tok, CONFIG["max_length"], label_enc), batch_size=CONFIG["eval_batch_size"], shuffle=False, **lkw)
    model = Encoder(name, NUM_CLASSES, CONFIG["dropout"], CONFIG["head_hidden_dim"], CONFIG["n_msd"]).to(device)
    model.load_state_dict(torch.load(f"{sd}/best_model.pt", map_location=device, weights_only=True))
    vm = evaluate(model, vl)["macro_f1"]; tm = evaluate(model, tel)
    torch.save(get_logits(model, vl),  f"{sd}/val_logits.pt")
    torch.save(get_logits(model, tel), f"{sd}/test_logits.pt")
    json.dump({"model":"muril","seed":42,"best_val_macro_f1":round(vm,4),"test_metrics":tm},
              open(f"{sd}/results.json","w"), indent=2)
    print("✅ salvaged muril_seed42:", tm); del model; torch.cuda.empty_cache()

In [54]:
# ── Run all ─────────────────────────────────────────────────────────────────
all_results = []
for mk in RUN_MODELS:
    for s in RUN_SEEDS:
        sd = f"{OUTPUT_DIR}/{mk}_seed{s}"
        if not FORCE_RETRAIN and os.path.exists(f"{sd}/results.json"):
            print(f"⏭  {mk} seed={s} done"); all_results.append(json.load(open(f"{sd}/results.json"))); continue
        print(f"\n▶ {mk} seed={s}")
        try: all_results.append(train_one(mk, CONFIG["models"][mk], s))
        except Exception as e:
            import traceback; print(f"❌ {mk} seed={s}: {e}"); traceback.print_exc()
        torch.cuda.empty_cache()
print(f"\n🎉 {len(all_results)}/{len(RUN_MODELS)*len(RUN_SEEDS)} runs")

⏭  banglishbert seed=42 done

▶ banglishbert seed=123


You're using a ElectraTokenizerFast tokenizer. Please note that with a fast tokenizer, using the `__call__` method is faster than using a method to encode the text followed by a call to the `pad` method to get a padded encoding.


  Ep 1/8 loss=0.8667 val_macroF1=0.7637 3.2min ✅BEST
  Ep 2/8 loss=0.6113 val_macroF1=0.7853 3.1min ✅BEST
  Ep 3/8 loss=0.5161 val_macroF1=0.7990 3.2min ✅BEST
  Ep 4/8 loss=0.4232 val_macroF1=0.8006 3.4min ✅BEST
  Ep 5/8 loss=0.3415 val_macroF1=0.8005 3.2min
  Ep 6/8 loss=0.2830 val_macroF1=0.7957 3.1min
  Ep 7/8 loss=0.2431 val_macroF1=0.7916 3.2min
  early stop ep7
  → [20% TEST] macroF1=0.7996 wF1=0.8124 acc=0.8161 mcc=0.7151

▶ banglishbert seed=456


You're using a ElectraTokenizerFast tokenizer. Please note that with a fast tokenizer, using the `__call__` method is faster than using a method to encode the text followed by a call to the `pad` method to get a padded encoding.


  Ep 1/8 loss=0.8693 val_macroF1=0.7413 3.2min ✅BEST
  Ep 2/8 loss=0.6116 val_macroF1=0.7901 3.2min ✅BEST
  Ep 3/8 loss=0.5173 val_macroF1=0.7991 3.2min ✅BEST
  Ep 4/8 loss=0.4264 val_macroF1=0.8021 3.2min ✅BEST
  Ep 5/8 loss=0.3455 val_macroF1=0.7972 3.1min
  Ep 6/8 loss=0.2850 val_macroF1=0.7996 3.1min
  Ep 7/8 loss=0.2463 val_macroF1=0.8005 3.2min
  early stop ep7
  → [20% TEST] macroF1=0.8083 wF1=0.8200 acc=0.8213 mcc=0.7261

▶ banglabert seed=42
  script-aware: banglabert uses Bangla-only scope | train=39,826, val=5,757/9,432, test=11,406/18,865


...

  Ep 1/8 loss=0.9647 val_macroF1=0.7626 1.9min ✅BEST
  Ep 2/8 loss=0.6501 val_macroF1=0.7942 1.9min ✅BEST
  Ep 3/8 loss=0.5399 val_macroF1=0.8081 2.0min ✅BEST
  Ep 4/8 loss=0.4417 val_macroF1=0.8021 1.9min
  Ep 5/8 loss=0.3546 val_macroF1=0.8033 1.9min
  Ep 6/8 loss=0.2940 val_macroF1=0.8041 1.9min
  early stop ep6
  → [Bangla-only 20% TEST] macroF1=0.8105 wF1=0.8178 acc=0.8177 mcc=0.7545

▶ banglabert seed=123
  script-aware: banglabert uses Bangla-only scope | train=39,826, val=5,757/9,432, test=11,406/18,865


You're using a ElectraTokenizerFast tokenizer. Please note that with a fast tokenizer, using the `__call__` method is faster than using a method to encode the text followed by a call to the `pad` method to get a padded encoding.


  Ep 1/8 loss=0.9852 val_macroF1=0.7632 1.9min ✅BEST
  Ep 2/8 loss=0.6492 val_macroF1=0.7891 1.9min ✅BEST
  Ep 3/8 loss=0.5403 val_macroF1=0.8042 1.9min ✅BEST
  Ep 4/8 loss=0.4422 val_macroF1=0.8087 1.8min ✅BEST
  Ep 5/8 loss=0.3562 val_macroF1=0.8040 1.9min
  Ep 6/8 loss=0.2947 val_macroF1=0.7993 1.9min
  Ep 7/8 loss=0.2560 val_macroF1=0.7999 1.8min
  early stop ep7
  → [Bangla-only 20% TEST] macroF1=0.8154 wF1=0.8211 acc=0.8218 mcc=0.7603

▶ banglabert seed=456
  script-aware: banglabert uses Bangla-only scope | train=39,826, val=5,757/9,432, test=11,406/18,865


You're using a ElectraTokenizerFast tokenizer. Please note that with a fast tokenizer, using the `__call__` method is faster than using a method to encode the text followed by a call to the `pad` method to get a padded encoding.


  Ep 1/8 loss=1.0124 val_macroF1=0.7545 1.9min ✅BEST
  Ep 2/8 loss=0.6562 val_macroF1=0.7907 2.0min ✅BEST
  Ep 3/8 loss=0.5445 val_macroF1=0.8079 1.9min ✅BEST
  Ep 4/8 loss=0.4453 val_macroF1=0.8050 1.9min
  Ep 5/8 loss=0.3554 val_macroF1=0.8050 1.9min
  Ep 6/8 loss=0.2936 val_macroF1=0.8017 1.9min
  early stop ep6
  → [Bangla-only 20% TEST] macroF1=0.8149 wF1=0.8205 acc=0.8216 mcc=0.7600
⏭  muril seed=42 done
⏭  muril seed=123 done
⏭  muril seed=456 done
⏭  xlmr seed=42 done
⏭  xlmr seed=123 done
⏭  xlmr seed=456 done

🎉 12/12 runs


In [55]:
# ── Per-run summary (20% test) ──────────────────────────────────────────────
if all_results:
    rows = [{"model": r["model"], "seed": r["seed"], **r["test_metrics"]} for r in all_results]
    sdf = pd.DataFrame(rows)
    pd.set_option("display.width", 200)
    print(sdf.to_string(index=False))
    print("\n── mean ± std per model ──")
    agg = sdf.groupby("model")[["macro_f1","weighted_f1","accuracy","mcc","auroc"]].agg(["mean","std"]).round(4)
    print(agg.to_string())
    sdf.to_csv(f"{OUTPUT_DIR}/per_run_summary.csv", index=False)
    print(f"\n✅ saved {OUTPUT_DIR}/per_run_summary.csv")

       model  seed  macro_f1  weighted_f1  accuracy    mcc  auroc
banglishbert    42    0.8069       0.8201    0.8206 0.7252 0.9567
banglishbert   123    0.7996       0.8124    0.8161 0.7151 0.9547
banglishbert   456    0.8083       0.8200    0.8213 0.7261 0.9568
  banglabert    42    0.8105       0.8178    0.8177 0.7545 0.9602
  banglabert   123    0.8154       0.8211    0.8218 0.7603 0.9599
  banglabert   456    0.8149       0.8205    0.8216 0.7600 0.9615
       muril    42    0.8055       0.8177    0.8185 0.7231 0.9521
       muril   123    0.8084       0.8216    0.8222 0.7284 0.9502
       muril   456    0.8066       0.8198    0.8210 0.7259 0.9523
        xlmr    42    0.8052       0.8159    0.8161 0.7199 0.9538
        xlmr   123    0.8016       0.8144    0.8145 0.7166 0.9503
        xlmr   456    0.8035       0.8173    0.8181 0.7215 0.9501

── mean ± std per model ──
             macro_f1         weighted_f1         accuracy             mcc           auroc        
               

---
**Outputs per run:** `best_model.pt`, `val_logits.pt`, `test_logits.pt`, `results.json`.
NB06 optimises ensemble weights on the val logits and reports the fused ensemble on the **20% test**.
Toggle any of `use_fgm / use_rdrop / use_ema / use_balanced_sampler` off to generate NB08 ablation rows.
